# Layerwise Logit-Lens And Unembedding Viewer

## Experiment
Interactive layerwise unembedding viewer: browse puzzle positions, show board/candidates, and inspect top-token probabilities across layers.

In [ ]:
from pathlib import Path
import os, sys

# Allow running from repo root or from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))


In [ ]:
CACHE_PATH = "results/3M-backtracking-packing/activations.npz"
CKPT_DIR   = "results/3M-backtracking-packing/checkpoint"

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib.widgets import TextBox, Button

plt.rcParams['font.family'] = ['Avenir']

from sudoku.probes.session import ProbeSession
from sudoku.activations import load_checkpoint
from sudoku.probes.activations import build_grid_at_step
from sudoku.data import decode_fill, SEP_TOKEN, PAD_TOKEN
from sudoku.data_bt import PUSH_TOKEN, POP_TOKEN, SUCCESS_TOKEN, PAD_TOKEN_BT

session            = ProbeSession(CACHE_PATH)
params, model_inst = load_checkpoint(CKPT_DIR)
cfg                = model_inst.config

n_layers    = cfg.n_layers
d_model     = cfg.d_model
vocab_size  = cfg.vocab_size
max_seq_len = cfg.max_seq_len
n_puzzles   = session.n_puzzles

print(f"Model : {n_layers} layers, d_model={d_model}, vocab_size={vocab_size}, max_seq_len={max_seq_len}")
print(f"Dataset: {n_puzzles} puzzles")


In [ ]:
%matplotlib widget

# ── Unembedding/readout helpers ──────────────────────────────────────────────
from functools import partial
from scripts.analysis.model_readouts import final_logits_np, softmax_fill as _softmax_fill
from scripts.analysis.sudoku_state import sequence_length as _seq_len, token_label as _tok_label

_lm_W = np.array(params["lm_head"]["kernel"], dtype=np.float32)  # (d_model, vocab_size)
_lm_b = np.array(params["lm_head"]["bias"], dtype=np.float32)    # (vocab_size,)


_unembed = partial(final_logits_np, params=params)

# ── JIT forward pass ──────────────────────────────────────────────────────────
@jax.jit
def _fwd(tokens_arr):
    """tokens_arr: (1, max_seq_len) → (logits, intermediates)."""
    return model_inst.apply({"params": params}, tokens_arr, return_intermediates=True)

# Forward-pass cache (re-runs only when puzzle changes)
_cache: dict = {}

def _ensure_cached(pi: int) -> None:
    if _cache.get("pi") == pi:
        return
    seq = session.sequences[pi]
    arr = np.full(max_seq_len, int(PAD_TOKEN_BT), dtype=np.int32)
    arr[:len(seq)] = seq
    logits_out, intermediates = _fwd(jnp.array(arr)[None])
    _cache.update({
        "pi":     pi,
        "logits": np.array(logits_out[0], dtype=np.float32),        # (T, vocab_size)
        "resids": np.stack([                                          # (n_layers, T, d_model)
            np.array(intermediates[f"layer_{l}_post_mlp"][0], dtype=np.float32)
            for l in range(n_layers)
        ]),
    })

def _get_probs(pi: int, pos: int):
    """Returns all_probs (n_layers+1, 729) and final_logits (vocab_size,).
    Row 0 of all_probs is the raw embedding; rows 1..n are post-MLP layers."""
    _ensure_cached(pi)
    seq = session.sequences[pi]

    emb_tok = np.array(params["token_emb"]["embedding"][int(seq[pos])], dtype=np.float32)
    emb_pos = np.array(params["pos_emb"]["embedding"][pos],             dtype=np.float32)
    emb_at  = (emb_tok + emb_pos)[None]                               # (1, d_model)

    layer_resids = _cache["resids"][:, pos, :]                         # (n_layers, d_model)
    all_resids   = np.concatenate([emb_at, layer_resids], axis=0)      # (n_layers+1, d_model)
    return _softmax_fill(_unembed(all_resids)), _cache["logits"][pos]


In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.widgets import TextBox, Button, Slider

TOP_K = 30

from scripts.analysis.sudoku_state import (
    board_after_position,
    candidates as _candidate_masks,
    hidden_singles as _hidden_single_placements,
)


def _compute_candidates(board):
    masks = _candidate_masks(board)
    return {
        cell: {d + 1 for d in range(9) if mask & (1 << d)}
        for cell, mask in enumerate(masks)
        if mask
    }


def _hidden_singles(candidates):
    masks = [0] * 81
    for cell, digits in candidates.items():
        for digit in digits:
            masks[cell] |= 1 << (digit - 1)
    return {(p.cell, p.digit) for p in _hidden_single_placements(masks)}

def _draw_board(ax, board, title=""):
    ax.clear()
    ax.set_xlim(-0.5, 8.5); ax.set_ylim(-0.5, 8.5)
    ax.set_aspect("equal"); ax.axis("off")
    cands        = _compute_candidates(board)
    hidden       = _hidden_singles(cands)
    hidden_cells = {cell for cell, _ in hidden}
    for idx in range(81):
        r, c = divmod(idx, 9); y0 = 8 - r
        if idx in board:
            ax.add_patch(plt.Rectangle((c-.5, y0-.5), 1., 1., color="#d0d0d0", zorder=1, linewidth=0))
            ax.text(c, y0, str(board[idx]), ha="center", va="center", fontsize=12, fontweight="bold", zorder=2)
        else:
            ax.add_patch(plt.Rectangle((c-.5, y0-.5), 1., 1.,
                                       color="#ffe8e8" if idx in hidden_cells else "white",
                                       zorder=1, linewidth=0))
            for d in cands.get(idx, ()):
                dr, dc = divmod(d-1, 3)
                is_h = (idx, d) in hidden
                ax.text(c+(dc-1)*0.32, y0+(1-dr)*0.30, str(d), ha="center", va="center",
                        fontsize=5.5, zorder=2,
                        color="red" if is_h else "#555555",
                        fontweight="bold" if is_h else "normal")
    for i in range(10):
        lw = 2.2 if i % 3 == 0 else 0.5
        ax.axhline(i-.5, color="black", lw=lw, zorder=3)
        ax.axvline(i-.5, color="black", lw=lw, zorder=3)
    if title: ax.set_title(title, fontsize=9)

def _board_dict(seq, pos):
    return board_after_position(seq, pos)

def _validity(board, cell, digit):
    if cell in board: return False
    r, c = divmod(cell, 9); br, bc = (r//3)*3, (c//3)*3
    for j in range(9):
        if board.get(r*9+j) == digit or board.get(j*9+c) == digit: return False
    for dr in range(3):
        for dc in range(3):
            if board.get((br+dr)*9+(bc+dc)) == digit: return False
    return True

def _get_probs(pi, pos):
    _ensure_cached(pi)
    seq = session.sequences[pi]
    emb = (np.array(params["token_emb"]["embedding"][int(seq[pos])], dtype=np.float32)
         + np.array(params["pos_emb"]["embedding"][pos],             dtype=np.float32))
    all_resids = np.concatenate([emb[None], _cache["resids"][:, pos, :]], axis=0)
    logits_ln  = _unembed(all_resids)
    logits_nln = all_resids.astype(np.float32) @ _lm_W + _lm_b
    probs      = _softmax_fill(logits_ln)
    return probs, _cache["logits"][pos], logits_ln[:, :729], logits_nln[:, :729], logits_ln[:, 729:], logits_nln[:, 729:]

# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 13))
ax_info   = fig.add_axes([0.02, 0.97, 0.96, 0.02]);  ax_info.axis("off")
info_txt  = ax_info.text(0.0, 0.5, "", fontsize=9, va="center")

ax_board  = fig.add_axes([0.02, 0.16, 0.24, 0.79])
ax_sfmx   = fig.add_axes([0.34, 0.52, 0.55, 0.43])
ax_cbar_s = fig.add_axes([0.9,  0.52, 0.01, 0.43])

# Bottom row: logit-lens line plot + per-layer delta bars
ax_lens   = fig.add_axes([0.30, 0.12, 0.42, 0.33])
ax_delta  = fig.add_axes([0.75, 0.12, 0.12, 0.33])

ax_slider = fig.add_axes([0.10, 0.07, 0.78, 0.025])
ax_puz    = fig.add_axes([0.10, 0.01, 0.10, 0.04])
ax_btn    = fig.add_axes([0.22, 0.01, 0.07, 0.04])

pos_slider = Slider(ax_slider, "Pos", 0, max_seq_len-1, valinit=25, valstep=1)
txt_puz    = TextBox(ax_puz, "Puzzle:", initial="0")
btn_go     = Button(ax_btn, "Go", color="#cce8ff", hovercolor="#99d1ff")

state = {"pi": 24}

def _update_slider_range(slen):
    pos_slider.valmax = slen - 1
    pos_slider.ax.set_xlim(0, slen - 1)
    if pos_slider.val >= slen:
        pos_slider.set_val(0)

# ── Render ────────────────────────────────────────────────────────────────────
# Tableau colors for the top-K lines (non-correct tokens)
_COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]

def _render(_=None):
    pi  = state["pi"]
    pos = int(round(pos_slider.val))
    seq  = session.sequences[pi]
    slen = _seq_len(seq)
    pos  = max(0, min(pos, slen - 1))

    sol      = session.solutions[pi] if session.solutions is not None else None
    board    = _board_dict(seq, pos)
    cur_lbl  = _tok_label(int(seq[pos]))
    next_tok = int(seq[pos+1]) if pos+1 < slen else -1
    next_lbl = _tok_label(next_tok) if next_tok >= 0 else "—"

    _draw_board(ax_board, board, f"Puzzle {pi}  |  pos {pos}: {cur_lbl}\nnext -> {next_lbl}")

    all_probs, _, logits_ln, logits_nln, control_ln, control_nln = _get_probs(pi, pos)
    top_idx    = np.argsort(all_probs[-1])[::-1][:TOP_K]
    col_labels = ["Emb"] + [f"L{l}" for l in range(n_layers)]
    x          = np.arange(len(col_labels))

    # Shared y-axis labels / colors for heatmap
    row_labels, tick_colors = [], []
    for k in range(TOP_K):
        tok = int(top_idx[k]); r, c, d = decode_fill(tok); cell = r*9+c
        valid   = _validity(board, cell, d)
        correct = sol is not None and int(sol[cell]) == d
        v_str = "v=+" if valid else "v=-"
        c_str = ("c=+" if correct else "c=-") if sol is not None else "c=?"
        row_labels.append(f"{'-> ' if tok == next_tok else ''}{_tok_label(tok)}  {v_str} {c_str}")
        tick_colors.append(
            "tab:red"    if tok == next_tok else
            "tab:green"  if valid and (sol is None or correct) else
            "tab:orange" if valid else "#999999"
        )

    # ── Softmax heatmap (top) ─────────────────────────────────────────────────
    hmap = all_probs[:, top_idx].T
    ax_sfmx.clear()
    im_s = ax_sfmx.imshow(hmap, aspect="auto", cmap="YlOrRd",
                          vmin=0, vmax=max(hmap.max(), 1e-6), interpolation="nearest")
    ax_sfmx.set_xticks(x);  ax_sfmx.set_xticklabels(col_labels, fontsize=9)
    ax_sfmx.set_yticks(range(TOP_K));  ax_sfmx.set_yticklabels(row_labels, fontsize=8)
    for tick, col in zip(ax_sfmx.get_yticklabels(), tick_colors): tick.set_color(col)
    ax_sfmx.set_xlabel("Layer", fontsize=9)
    ax_sfmx.set_title(
        f"Softmax probability  |  top {TOP_K} by last layer  |  "
        "-> = correct  green=valid+correct  orange=valid+wrong  gray=invalid", fontsize=8)
    if next_tok >= 0 and next_tok in top_idx:
        yi = int(np.where(top_idx == next_tok)[0][0])
        ax_sfmx.axhspan(yi-.5, yi+.5, color="steelblue", alpha=0.15, zorder=0)
    for row in range(TOP_K):
        for col in range(len(col_labels)):
            v = hmap[row, col]
            if v > 0.02:
                ax_sfmx.text(col, row, f"{v:.2f}", ha="center", va="center",
                             fontsize=6, color="black" if v < 0.45 else "white")
    ax_cbar_s.clear();  plt.colorbar(im_s, cax=ax_cbar_s, label="P(token)")

    # ── Logit-lens line plot (bottom-left) ────────────────────────────────────
    ax_lens.clear()
    color_idx = 0
    for k in range(TOP_K):
        tok = int(top_idx[k])
        y   = logits_ln[:, tok]           # trajectory with LN
        y0  = logits_nln[:, tok]          # trajectory without LN
        is_next = tok == next_tok
        if is_next:
            ax_lens.plot(x, y,  color="tab:red", lw=2.2, zorder=4, label="w/ LN")
            # ax_lens.plot(x, y0, color="tab:red", lw=1.2, zorder=4, ls="--", alpha=0.6, label="w/o LN")
        else:
            col = _COLORS[color_idx % len(_COLORS)]
            ax_lens.plot(x, y,  color=col, lw=0.9, alpha=0.55, zorder=2)
            # ax_lens.plot(x, y0, color=col, lw=0.6, alpha=0.30, zorder=2, ls="--")
            color_idx += 1
    
    ax_lens.plot(x, control_ln[:, 2], lw=2.2, color="tab:purple", label="[PUSH]")
    ax_lens.plot(x, control_ln[:, 3], lw=2.2, color="tab:orange", label="[POP]")
    ax_lens.plot(x, control_ln[:, 4], lw=2.2, color="tab:green", label="[SUCCESS]")


    ax_lens.set_xticks(x);  ax_lens.set_xticklabels(col_labels, fontsize=8)
    ax_lens.set_ylabel("Logit score", fontsize=8)
    ax_lens.set_title(
        "Logit lens  —  solid: with LN  |  dashed: without LN  |  red = correct next token",
        fontsize=8)
    ax_lens.axvline(len(col_labels) - 1, color="#aaaaaa", lw=0.8, ls=":")
    ax_lens.grid(axis="y", lw=0.4, alpha=0.4)

    # ── Per-layer Δlogit bar chart (bottom-right) ─────────────────────────────
    # delta[l] = logit after block l  −  logit before block l
    # shape: (n_layers, TOP_K) — row 0 is Emb→L0, row l is L{l-1}→L{l}
    deltas = np.diff(logits_ln[:, top_idx], axis=0)   # (n_layers, TOP_K)
    y_bars = np.arange(n_layers)

    ax_delta.clear()
    bar_h  = 0.35
    # top-1 token (correct next or highest-prob)
    ax_delta.barh(y_bars - bar_h/2, deltas[:, 0],
                  height=bar_h, color="tab:red" if int(top_idx[0]) == next_tok else "tab:blue",
                  label=_tok_label(int(top_idx[0])))
    # top-2 token
    ax_delta.barh(y_bars + bar_h/2, deltas[:, 1],
                  height=bar_h, color="#aaaaaa", alpha=0.7,
                  label=_tok_label(int(top_idx[1])))

    ax_delta.axvline(0, color="black", lw=0.8)
    ax_delta.set_yticks(y_bars)
    ax_delta.set_yticklabels([f"L{l}" for l in range(n_layers)], fontsize=8)
    ax_delta.set_xlabel("Δlogit", fontsize=8)
    ax_delta.set_title("Δlogit\nper block", fontsize=8)
    ax_delta.legend(fontsize=6, loc="lower right")
    ax_delta.grid(axis="x", lw=0.4, alpha=0.4)

    p_correct = float(all_probs[-1, next_tok]) if 0 <= next_tok <= 728 else float("nan")
    info_txt.set_text(
        f"Puzzle {pi}  |  seq len {slen}  |  pos {pos}: {cur_lbl}  →  next: {next_lbl}"
        + (f"  |  P(correct) @ last layer: {p_correct:.3f}" if not np.isnan(p_correct) else ""))

    fig.canvas.draw_idle()


def _on_puzzle_submit(_=None):
    try:
        pi = int(txt_puz.text.strip()) % n_puzzles
        state["pi"] = pi
        _update_slider_range(_seq_len(session.sequences[pi]))
        _render()
    except ValueError:
        pass

pos_slider.on_changed(_render)
txt_puz.on_submit(_on_puzzle_submit)
btn_go.on_clicked(_on_puzzle_submit)

_update_slider_range(_seq_len(session.sequences[state["pi"]]))
_render()
plt.show()

